# Event Web Scraping – Version 5 (final)

Scrapes dates and attendance for **Street Parade** and **Züri Fäscht** (2019–2025) and saves one row per event day to `data/processed/zurich_event_days.csv`.

## 1. Imports

In [1]:
import os
import re
import time
from urllib import robotparser

import requests
from bs4 import BeautifulSoup
import pandas as pd

myheaders = {"User-Agent": "Mozilla/5.0 (compatible; HSLU-CIP-student-project)"}

`get_page()` checks `robots.txt` first and only downloads the page if scraping is allowed.
If `robots.txt` cannot be read, the page is skipped (only a missing `robots.txt`, status 404, counts as "no restrictions").

In [2]:
def get_page(url):
    site = "/".join(url.split("/")[:3])
    try:
        robots = requests.get(site + "/robots.txt", headers=myheaders, timeout=20)
        if robots.status_code == 200:
            rp = robotparser.RobotFileParser()
            rp.parse(robots.text.splitlines())
            if not rp.can_fetch(myheaders["User-Agent"], url):
                print("Not allowed by robots.txt:", url)
                return None
        elif robots.status_code != 404:
            print("robots.txt could not be checked, page skipped:", url)
            return None
        page = requests.get(url, headers=myheaders, timeout=20)
    except requests.RequestException:
        print("Request failed:", url)
        return None
    time.sleep(1)
    if page.status_code != 200:
        print("Status", page.status_code, ":", url)
        return None
    return BeautifulSoup(page.text, "html.parser")

## 2. Street Parade

**Attendance and cancellations** – official history page. Each edition starts with an `<h3>` year heading, followed by e.g. `850'000 Visitors`, or by `Abgesagt` (canceled).

In [3]:
sp_url = "https://www.streetparade.com/en/historie"
soup = get_page(sp_url)

street_parade = {}
if soup is not None:
    for h3 in soup.find_all("h3"):
        year = h3.getText(strip=True)
        if year not in ["2019", "2020", "2021", "2022", "2023", "2024", "2025"]:
            continue
        info = {"status": "held", "attendance": None}
        # read the texts after the heading until the next year heading starts
        for text in h3.find_all_next(string=True):
            if text.parent.name == "h3" and text.parent != h3:
                break
            if text.strip() == "Abgesagt":
                info["status"] = "canceled"
            if text.strip().endswith("Visitors"):
                info["attendance"] = int(text.split()[0].replace("'", ""))
        street_parade[int(year)] = info

print(street_parade)

{2019: {'status': 'held', 'attendance': 850000}, 2020: {'status': 'canceled', 'attendance': None}, 2021: {'status': 'canceled', 'attendance': None}, 2022: {'status': 'held', 'attendance': 900000}, 2023: {'status': 'held', 'attendance': 920000}, 2024: {'status': 'held', 'attendance': 920000}, 2025: {'status': 'held', 'attendance': 800000}}


**Dates** are not on the history page.

* 2019 – date tag of a City of Zürich police report published during the parade; the report states *Die Parade dauert zurzeit noch an* (the parade is still running)
* 2022–2025 – traffic notice of the Stadtpolizei Zürich, which states the date in one sentence:
  *Am Samstag, 13. August 2022, findet die Street Parade statt.*
  2022 is read from stadt-zuerich.ch; for 2023–2025 the City page could not be found,
  so verbatim copies on police news sites are used (they name the Stadtpolizei as source).

In [4]:
sp_dates = {}   # year -> (date, source)

url = "https://www.stadt-zuerich.ch/misc/de/mitteilungsarchiv/medienmitteilungen/2019/08/street_parade_2019zwischenbilanzderstadtpolizeizuerich.html"
soup = get_page(url)
if soup is not None:
    tag = soup.find("meta", attrs={"name": "date"})
    text = " ".join(soup.getText(" ").split())
    # live police report: the parade was still running when it was published
    if tag is not None and "Die Parade dauert zurzeit noch an" in text:
        sp_dates[2019] = (tag["content"][:10], url)
        print(2019, "found:", tag["content"][:10], "- report says: Die Parade dauert zurzeit noch an")
    else:
        print(2019, "date NOT confirmed on", url)

traffic_pages = {
    2022: "https://www.stadt-zuerich.ch/de/aktuell/medienmitteilungen/2022/08/220809a.html",
    2023: "https://polizei.news/2023/08/09/stadt-zuerich-zh-verkehrsvorschriften-street-parade-2023/",
    2024: "https://www.polizeinews.ch/2024/08/06/stadt-zuerich-zh-verkehrsvorschriften-street-parade-2024/",
    2025: "https://polizeiticker.ch/artikel/street-parade-2025-in-zuerich-sperrungen-und-umleitungen-im-stadtzentrum-246738",
}

for year, url in traffic_pages.items():
    soup = get_page(url)
    if soup is None:
        continue
    text = " ".join(soup.getText(" ").split())   # also removes non-breaking spaces
    match = re.search(r"Am Samstag, (\d{1,2})\. August " + str(year) + ", findet die Street Parade", text)
    if match:
        sp_dates[year] = (str(year) + "-08-" + match.group(1).zfill(2), url)
        print(year, "found:", match.group(0))
    else:
        print(year, "date sentence NOT found on", url)

print(sp_dates)

2019 found: 2019-08-10 - report says: Die Parade dauert zurzeit noch an
2022 found: Am Samstag, 13. August 2022, findet die Street Parade
2023 found: Am Samstag, 12. August 2023, findet die Street Parade
2024 found: Am Samstag, 10. August 2024, findet die Street Parade
2025 found: Am Samstag, 9. August 2025, findet die Street Parade
{2019: ('2019-08-10', 'https://www.stadt-zuerich.ch/misc/de/mitteilungsarchiv/medienmitteilungen/2019/08/street_parade_2019zwischenbilanzderstadtpolizeizuerich.html'), 2022: ('2022-08-13', 'https://www.stadt-zuerich.ch/de/aktuell/medienmitteilungen/2022/08/220809a.html'), 2023: ('2023-08-12', 'https://polizei.news/2023/08/09/stadt-zuerich-zh-verkehrsvorschriften-street-parade-2023/'), 2024: ('2024-08-10', 'https://www.polizeinews.ch/2024/08/06/stadt-zuerich-zh-verkehrsvorschriften-street-parade-2024/'), 2025: ('2025-08-09', 'https://polizeiticker.ch/artikel/street-parade-2025-in-zuerich-sperrungen-und-umleitungen-im-stadtzentrum-246738')}


## 3. Züri Fäscht

The official website (zuerifaescht.ch) does not allow scraping in its `robots.txt`, so other sources are used.

Both date pages state the range in their `<meta name="description">` tag:

* 2019 – Zürcher Handelskammer: *vom Freitag, 5. Juli bis Sonntag, 7. Juli 2019*
* 2023 – freizeit.ch: *vom 7. bis 9. Juli 2023*

In [5]:
zf_pages = {
    2019: "https://www.zhk.ch/de/wirtschaft-und-politik/mobilitaet-und-infrastruktur/verkehrskonzept-zueri-faescht-2019.html",
    2023: "https://www.freizeit.ch/kalender/zuri-fascht/",
}

zueri_faescht = {}
for year, url in zf_pages.items():
    zueri_faescht[year] = {"start": None, "end": None, "date_source": None,
                           "attendance": None, "attendance_source": None}
    soup = get_page(url)
    if soup is None:
        continue
    tag = soup.find("meta", attrs={"name": "description"})
    if tag is None:
        continue
    # first day, any words in between, last day: "5. Juli bis Sonntag, 7. Juli 2019"
    match = re.search(r"(\d{1,2})\.\D+(\d{1,2})\. Juli " + str(year), tag["content"])
    if match:
        zueri_faescht[year]["start"] = str(year) + "-07-" + match.group(1).zfill(2)
        zueri_faescht[year]["end"] = str(year) + "-07-" + match.group(2).zfill(2)
        zueri_faescht[year]["date_source"] = url

**Attendance** (organisers' estimates for the whole festival):

* 2019 – German Wikipedia visitor table, row *2019 | 2,5 Millionen*
* 2023 – swissinfo.ch report after the festival: *The estimated two million … organisers said*

In [6]:
url = "https://de.wikipedia.org/wiki/Z%C3%BCri_F%C3%A4scht"
soup = get_page(url)
if soup is not None:
    for row in soup.find_all("tr"):
        cells = [cell.getText(strip=True) for cell in row.find_all(["th", "td"])]
        if len(cells) >= 2 and cells[0] == "2019" and "Millionen" in cells[1]:
            millions = float(cells[1].split()[0].replace(",", "."))
            zueri_faescht[2019]["attendance"] = int(millions * 1_000_000)
            zueri_faescht[2019]["attendance_source"] = url

url = "https://www.swissinfo.ch/eng/society/two-million-people-flock-to-three-day-zurich-festival/48653204"
soup = get_page(url)
words = {"one": 1, "two": 2, "three": 3}
if soup is not None:
    for p in soup.find_all("p"):
        match = re.search(r"(\w+) million", p.getText())
        if "organisers said" in p.getText() and match and match.group(1).lower() in words:
            zueri_faescht[2023]["attendance"] = words[match.group(1).lower()] * 1_000_000
            zueri_faescht[2023]["attendance_source"] = url

print(zueri_faescht)

{2019: {'start': '2019-07-05', 'end': '2019-07-07', 'date_source': 'https://www.zhk.ch/de/wirtschaft-und-politik/mobilitaet-und-infrastruktur/verkehrskonzept-zueri-faescht-2019.html', 'attendance': 2500000, 'attendance_source': 'https://de.wikipedia.org/wiki/Z%C3%BCri_F%C3%A4scht'}, 2023: {'start': '2023-07-07', 'end': '2023-07-09', 'date_source': 'https://www.freizeit.ch/kalender/zuri-fascht/', 'attendance': 2000000, 'attendance_source': 'https://www.swissinfo.ch/eng/society/two-million-people-flock-to-three-day-zurich-festival/48653204'}}


## 4. Cleaning

In [7]:
rows = []
for year, info in street_parade.items():
    date, date_url = sp_dates.get(year, (None, None))
    rows.append({"event_name": "Street Parade", "event_year": year, "status": info["status"],
                 "start": date, "end": date, "edition_attendance": info["attendance"],
                 "date_source": date_url, "attendance_source": sp_url})
for year, info in zueri_faescht.items():
    rows.append({"event_name": "Züri Fäscht", "event_year": year, "status": "held",
                 "start": info["start"], "end": info["end"], "edition_attendance": info["attendance"],
                 "date_source": info["date_source"], "attendance_source": info["attendance_source"]})

editions = pd.DataFrame(rows)
editions["start"] = pd.to_datetime(editions["start"])   # all dates are YYYY-MM-DD
editions["end"] = pd.to_datetime(editions["end"])
editions["edition_attendance"] = pd.to_numeric(editions["edition_attendance"]).astype("Int64")
editions = editions.drop_duplicates(subset=["event_name", "event_year"])
editions

,event_name,event_year,status,start,end,edition_attendance,date_source,attendance_source
0,Street Parade,2019,held,2019-08-10,2019-08-10,850000,https://www.stadt-zuerich.ch/misc/de/mitteilun...,https://www.streetparade.com/en/historie
1,Street Parade,2020,canceled,NaT,NaT,<NA>,NaN,https://www.streetparade.com/en/historie
2,Street Parade,2021,canceled,NaT,NaT,<NA>,NaN,https://www.streetparade.com/en/historie
3,Street Parade,2022,held,2022-08-13,2022-08-13,900000,https://www.stadt-zuerich.ch/de/aktuell/medien...,https://www.streetparade.com/en/historie
4,Street Parade,2023,held,2023-08-12,2023-08-12,920000,https://polizei.news/2023/08/09/stadt-zuerich-...,https://www.streetparade.com/en/historie
5,Street Parade,2024,held,2024-08-10,2024-08-10,920000,https://www.polizeinews.ch/2024/08/06/stadt-zu...,https://www.streetparade.com/en/historie
6,Street Parade,2025,held,2025-08-09,2025-08-09,800000,https://polizeiticker.ch/artikel/street-parade...,https://www.streetparade.com/en/historie
7,Züri Fäscht,2019,held,2019-07-05,2019-07-07,2500000,https://www.zhk.ch/de/wirtschaft-und-politik/m...,https://de.wikipedia.org/wiki/Z%C3%BCri_F%C3%A...
8,Züri Fäscht,2023,held,2023-07-07,2023-07-09,2000000,https://www.freizeit.ch/kalender/zuri-fascht/,https://www.swissinfo.ch/eng/society/two-milli...


## 5. Daily rows

Only held editions with a scraped date become event days.

* `edition_attendance` – estimate for the **whole edition** (repeated on each day)
* `estimated_daily_attendance` – edition attendance divided by the number of event days. This assumes equal attendance on every day; it is **not** measured daily attendance.

In [8]:
day_rows = []
for _, ed in editions.iterrows():
    if ed["status"] != "held" or pd.isna(ed["start"]) or pd.isna(ed["end"]):
        continue
    days = pd.date_range(ed["start"], ed["end"])
    for number, day in enumerate(days, start=1):
        day_rows.append({"date": day, "event_name": ed["event_name"], "event_year": ed["event_year"],
                         "edition_attendance": ed["edition_attendance"],
                         "estimated_daily_attendance": ed["edition_attendance"] / len(days),
                         "event_day_number": number,
                         "date_source": ed["date_source"], "attendance_source": ed["attendance_source"]})

columns = ["date", "event_name", "event_year", "edition_attendance", "estimated_daily_attendance",
           "event_day_number", "date_source", "attendance_source"]
event_days = pd.DataFrame(day_rows, columns=columns)
event_days["date"] = pd.to_datetime(event_days["date"])
event_days["estimated_daily_attendance"] = pd.to_numeric(event_days["estimated_daily_attendance"])
event_days = event_days.sort_values("date").reset_index(drop=True)
event_days

,date,event_name,event_year,edition_attendance,estimated_daily_attendance,event_day_number,date_source,attendance_source
0,2019-07-05,Züri Fäscht,2019,2500000,833333.333333,1,https://www.zhk.ch/de/wirtschaft-und-politik/m...,https://de.wikipedia.org/wiki/Z%C3%BCri_F%C3%A...
1,2019-07-06,Züri Fäscht,2019,2500000,833333.333333,2,https://www.zhk.ch/de/wirtschaft-und-politik/m...,https://de.wikipedia.org/wiki/Z%C3%BCri_F%C3%A...
2,2019-07-07,Züri Fäscht,2019,2500000,833333.333333,3,https://www.zhk.ch/de/wirtschaft-und-politik/m...,https://de.wikipedia.org/wiki/Z%C3%BCri_F%C3%A...
3,2019-08-10,Street Parade,2019,850000,850000.000000,1,https://www.stadt-zuerich.ch/misc/de/mitteilun...,https://www.streetparade.com/en/historie
4,2022-08-13,Street Parade,2022,900000,900000.000000,1,https://www.stadt-zuerich.ch/de/aktuell/medien...,https://www.streetparade.com/en/historie
5,2023-07-07,Züri Fäscht,2023,2000000,666666.666667,1,https://www.freizeit.ch/kalender/zuri-fascht/,https://www.swissinfo.ch/eng/society/two-milli...
6,2023-07-08,Züri Fäscht,2023,2000000,666666.666667,2,https://www.freizeit.ch/kalender/zuri-fascht/,https://www.swissinfo.ch/eng/society/two-milli...
7,2023-07-09,Züri Fäscht,2023,2000000,666666.666667,3,https://www.freizeit.ch/kalender/zuri-fascht/,https://www.swissinfo.ch/eng/society/two-milli...
8,2023-08-12,Street Parade,2023,920000,920000.000000,1,https://polizei.news/2023/08/09/stadt-zuerich-...,https://www.streetparade.com/en/historie
9,2024-08-10,Street Parade,2024,920000,920000.000000,1,https://www.polizeinews.ch/2024/08/06/stadt-zu...,https://www.streetparade.com/en/historie


## 6. Checks and export

The CSV is only saved if all expected event days were scraped (11 rows) and the checks pass.

In [9]:
problems = []
expected = {"Street Parade": [2019, 2022, 2023, 2024, 2025], "Züri Fäscht": [2019, 2023]}

for name, years in expected.items():
    found = sorted(event_days.loc[event_days["event_name"] == name, "event_year"].unique().tolist())
    if found != years:
        problems.append(name + ": found years " + str(found) + ", expected " + str(years))

sp_days = event_days[event_days["event_name"] == "Street Parade"]
if (sp_days["date"].dt.day_name() != "Saturday").any():
    problems.append("Street Parade date that is not a Saturday")

for year in expected["Züri Fäscht"]:
    days = event_days.loc[(event_days["event_name"] == "Züri Fäscht") & (event_days["event_year"] == year), "date"]
    if len(days) > 0 and days.dt.day_name().tolist() != ["Friday", "Saturday", "Sunday"]:
        problems.append("Züri Fäscht " + str(year) + " is not Friday to Sunday")

if event_days.duplicated(["date", "event_name"]).any():
    problems.append("duplicate event days")

# each edition has one attendance value, and daily estimate x days = edition attendance
for (name, year), days in event_days.groupby(["event_name", "event_year"]):
    if days["edition_attendance"].nunique() > 1:
        problems.append(name + " " + str(year) + ": different attendance values")
    total = days["estimated_daily_attendance"].sum()
    if days["edition_attendance"].notna().all() and abs(total - days["edition_attendance"].iloc[0]) > 0.01:
        problems.append(name + " " + str(year) + ": daily estimate does not add up")

# every date needs a source URL; attendance needs one where attendance exists
if event_days["date_source"].isna().any():
    problems.append("date without date_source")
if (event_days["edition_attendance"].notna() & event_days["attendance_source"].isna()).any():
    problems.append("attendance without attendance_source")

# do not save the CSV if attendance is missing
if event_days["edition_attendance"].isna().any():
    problems.append("missing attendance")

print("Event days:", len(event_days), "(expected 11)")
print("Missing attendance:", event_days.loc[event_days["edition_attendance"].isna(), "date"].dt.date.tolist())
print("Problems:", problems)

if len(event_days) == 11 and len(problems) == 0:
    os.makedirs("../../../data/processed", exist_ok=True)
    export = event_days.copy()
    export["date"] = export["date"].dt.strftime("%Y-%m-%d")
    export.to_csv("../../../data/processed/zurich_event_days.csv", index=False, encoding="utf-8-sig")   # utf-8-sig: Excel shows ü/ä correctly
    print("Saved data/processed/zurich_event_days.csv")
else:
    print("NOT saved - event days are missing or incorrect.")


Event days: 11 (expected 11)
Missing attendance: []
Problems: []
Saved data/processed/zurich_event_days.csv
